# EchoGesture 02: train, INT8 export and evaluate

**Session options:** Accelerator **GPU T4 x2** (or P100), Internet **ON**.  
**Inputs:** `echogesture` (newest version) and the **output of notebook 01** (Add Input, then Your Work, then the 01 notebook).

Run interactively for cells 1-4 (quick baseline), then use **Save & Run All** for the long pretraining.

In [ ]:
# code: copy the newest echogesture dataset into /kaggle/working (idempotent)
import os, glob, shutil, subprocess
W = '/kaggle/working/echogesture'
src = sorted(glob.glob('/kaggle/input/**/echogesture/src/main.py', recursive=True))
assert src, 'Add your echogesture dataset as an input'
shutil.rmtree(W, ignore_errors=True)
shutil.copytree(os.path.dirname(os.path.dirname(src[0])), W)
os.chdir(W); print(os.getcwd())
print('code version:', open('src/VERSION').read().strip() if os.path.exists('src/VERSION') else 'OLD (no VERSION file)')

In [ ]:
# no pip installs here: MediaPipe pulls protobuf 7.x, which can break Kaggle's TensorFlow.
# Training, INT8 export and landmark-level evaluation need only TF + numpy.
import tensorflow as tf, keras, numpy as np
print('TF', tf.__version__, '| Keras', keras.__version__, '| numpy', np.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# locate data from notebook 01 (its output is mounted under /kaggle/input)
hits = glob.glob('/kaggle/input/**/wlasl300.npz', recursive=True)
assert hits, 'Add the OUTPUT of 01_extract_landmarks as an input'
D = os.path.dirname(hits[0]); print(D)
!ls -la {D}

In [ ]:
# A. quick baseline: WLASL300 only, no pretraining (about 10-15 min on GPU)
!python src/training/train.py --data {D}/wlasl300.npz --vocab src/data/wlasl/vocab_wlasl300.json --out runs/base300 --epochs 80 --bs 64 --workers 4

In [ ]:
# B. pretrain on GISLR (250 signs, about 94k clips), about 1.5-2 h
!python src/training/train.py --data {D}/gislr.npz --out runs/pretrain --epochs 40 --bs 256 --lr 1e-3 --workers 4

In [ ]:
# C. intermediate stage: WLASL2000 (all glosses, about 8k train clips)
!python src/training/train.py --data {D}/wlasl2000.npz --vocab src/data/wlasl/vocab_wlasl2000.json --init runs/pretrain/best.keras --out runs/wl2000 --epochs 60 --bs 128 --workers 4

In [ ]:
# D. fine-tune on the target vocabulary (WLASL300; change to 100 if organisers say so)
!python src/training/train.py --data {D}/wlasl300.npz --vocab src/data/wlasl/vocab_wlasl300.json --init runs/wl2000/best.keras --out runs/ft300 --epochs 120 --bs 64 --workers 4
!cat runs/base300/val_report.json runs/ft300/val_report.json

In [ ]:
# E. INT8 export, install, evaluate, benchmark, results.csv
RUN = 'runs/ft300'
!python src/training/convert.py --ckpt {RUN}/best.keras --calib {RUN}/calib_windows.npy --out src/models/echogesture_int8.tflite
!cp {RUN}/vocab.json src/models/vocab.json
!python src/tools/evaluate.py --packed {D}/wlasl300.npz --split test --device 'Kaggle CPU'
!python src/tools/bench_latency.py --device 'Kaggle CPU'
!python src/tools/make_results_csv.py

In [ ]:
# F. bundle what we need back on the laptop
!zip -qr /kaggle/working/echogesture_trained.zip src/models runs/*/val_report.json runs/*/history.csv runs/*/best.keras runs/*/vocab.json results
!ls -lh /kaggle/working/echogesture_trained.zip